# N64 PIMC Minimization Visualization

This notebook visualizes the minimization trajectory of a 64-particle Helium PIMC configuration.
Run `minimize_N64.py` first to generate the trajectory file.

## Initialization

In [11]:
import sys
import os
import numpy as np
import pandas as pd

# Add parent directory to path to import from jax_landscape
sys.path.insert(0, os.path.abspath('../..'))

from jax_landscape.io.pimc import load_pimc_worldline_file

# Visualization libraries
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider, Output, HBox

## Load Trajectory and Log Data

In [12]:
# System parameters (from test_full_wl)
N = 64
n = 0.0218  # Angstrom^-3
L = (N/n)**(1/3)  # Box size
box = [L, L, L]

# Load minimization trajectory
trajectory_file = 'N64-cycle1.conf0.trajectory.dat'
log_file = 'N64-cycle1.conf0.log'

print(f"Loading trajectory from {trajectory_file}...")
confs = load_pimc_worldline_file(trajectory_file, Lx=box[0], Ly=box[1], Lz=box[2])
print(f"✅ Loaded {len(confs)} configurations")

# Load log data
print(f"\nLoading energy log from {log_file}...")
log_data = pd.read_csv(log_file,comment='#')
print(f"✅ Loaded {len(log_data)} log entries")
print(f"\nLog columns: {list(log_data.columns)}")
print(f"Iteration range: {log_data['Iteration'].min()} to {log_data['Iteration'].max()}")

# Get configuration indices (iteration numbers)
conf_indices = sorted(confs.keys())
print(f"\nConfiguration snapshots at iterations: {conf_indices[:5]}...{conf_indices[-3:]}")

Loading trajectory from N64-cycle1.conf0.trajectory.dat...
✅ Loaded 26 configurations

Loading energy log from N64-cycle1.conf0.log...
✅ Loaded 726 log entries

Log columns: ['Iteration', 'Energy(Urp)', 'E_sp', 'E_int', 'GradientNorm']
Iteration range: 0 to 708

Configuration snapshots at iterations: [0, 100, 200, 300, 400]...[680, 690, 700]


## Energy Evolution Plot

In [13]:
# Plot energy evolution
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Total Energy (Urp)', 'Spring Energy (E_sp)', 'Interaction Energy (E_int)'),
    vertical_spacing=0.1
)

# Total energy
fig.add_trace(
    go.Scatter(x=log_data['Iteration'], y=log_data['Energy(Urp)'], 
               mode='lines', name='Urp', line=dict(color='blue')),
    row=1, col=1
)

# Spring energy
fig.add_trace(
    go.Scatter(x=log_data['Iteration'], y=log_data['E_sp'], 
               mode='lines', name='E_sp', line=dict(color='green')),
    row=2, col=1
)

# Interaction energy
fig.add_trace(
    go.Scatter(x=log_data['Iteration'], y=log_data['E_int'], 
               mode='lines', name='E_int', line=dict(color='red')),
    row=3, col=1
)

# Add vertical lines for saved configurations
for iter_num in conf_indices:
    for row in [1, 2, 3]:
        fig.add_vline(x=iter_num, line_dash="dot", line_color="gray", opacity=0.3, row=row, col=1)

fig.update_xaxes(title_text="Iteration", row=3, col=1)
fig.update_yaxes(title_text="Energy (kB·K)", row=1, col=1)
fig.update_yaxes(title_text="Energy (kB·K)", row=2, col=1)
fig.update_yaxes(title_text="Energy (kB·K)", row=3, col=1)

fig.update_layout(height=800, showlegend=False, 
                  title_text="Energy Evolution During Minimization<br><sub>Vertical lines show saved trajectory snapshots</sub>")
fig.show()

# Print energy summary
print(f"\n📊 Energy Summary:")
print(f"Initial Urp: {log_data['Energy(Urp)'].iloc[0]:.2f} kB·K")
print(f"Final Urp: {log_data['Energy(Urp)'].iloc[-1]:.2f} kB·K")
print(f"Reduction: {log_data['Energy(Urp)'].iloc[0] - log_data['Energy(Urp)'].iloc[-1]:.2f} kB·K")
print(f"\nFinal E_sp: {log_data['E_sp'].iloc[-1]:.2f} kB·K")
print(f"Final E_int: {log_data['E_int'].iloc[-1]:.2f} kB·K")


📊 Energy Summary:
Initial Urp: 21617.53 kB·K
Final Urp: -3267.46 kB·K
Reduction: 24884.99 kB·K

Final E_sp: 0.25 kB·K
Final E_int: -3267.70 kB·K


## 3D Worldline Visualization

In [14]:
def plot_worldline_3d(config_idx):
    """
    Plot the 3D coordinate space trace of worldline path.
    Returns the figure object.
    """
    conf_iter = conf_indices[config_idx]
    conf = confs[conf_iter]
    
    # Get energy at this iteration
    log_row = log_data[log_data['Iteration'] == conf_iter]
    if len(log_row) > 0:
        energy_urp = log_row['Energy(Urp)'].values[0]
        energy_sp = log_row['E_sp'].values[0]
        energy_int = log_row['E_int'].values[0]
    else:
        energy_urp = energy_sp = energy_int = np.nan

    fig = go.Figure()

    Lx, Ly, Lz = box[0], box[1], box[2]
    
    # Wrap coordinates to [0, L]
    wrapped_coords = conf.beadCoord.copy()
    wrapped_coords[:, :, 0] = wrapped_coords[:, :, 0] % Lx
    wrapped_coords[:, :, 1] = wrapped_coords[:, :, 1] % Ly
    wrapped_coords[:, :, 2] = wrapped_coords[:, :, 2] % Lz

    # Get normalized cycle lengths
    cycle_lengths_normalized = conf.cycleSizeDist / conf.numTimeSlices
    unique_cycle_lengths = np.unique(cycle_lengths_normalized)

    # Create color mapping
    cycle_length_to_color_idx = {length: idx for idx, length in enumerate(unique_cycle_lengths)}
    num_unique_lengths = len(unique_cycle_lengths)

    # Group beads by cycle length
    beads_by_cycle_length = {length: {'x': [], 'y': [], 'z': []} for length in unique_cycle_lengths}

    for m in range(conf.numTimeSlices):
        for n in range(conf.numParticles):
            cycle_idx = conf.cycleIndex[m, n]
            cycle_length = cycle_lengths_normalized[cycle_idx]
            beads_by_cycle_length[cycle_length]['x'].append(wrapped_coords[m, n, 0])
            beads_by_cycle_length[cycle_length]['y'].append(wrapped_coords[m, n, 1])
            beads_by_cycle_length[cycle_length]['z'].append(wrapped_coords[m, n, 2])

    # Plot each cycle length group
    for cycle_length in unique_cycle_lengths:
        color_idx = cycle_length_to_color_idx[cycle_length]
        color = f'hsl({(color_idx * 360 / num_unique_lengths) % 360}, 80, 50)'

        coords = beads_by_cycle_length[cycle_length]
        fig.add_trace(go.Scatter3d(
            x=coords['x'],
            y=coords['y'],
            z=coords['z'],
            mode='markers',
            marker=dict(size=2, color=color),
            name=f'cycle {cycle_length:.0f}'
        ))

    # Draw box boundaries (thick black lines)
    box_edges = [
        # Bottom face
        ([0, Lx], [0, 0], [0, 0]),
        ([Lx, Lx], [0, Ly], [0, 0]),
        ([Lx, 0], [Ly, Ly], [0, 0]),
        ([0, 0], [Ly, 0], [0, 0]),
        # Top face
        ([0, Lx], [0, 0], [Lz, Lz]),
        ([Lx, Lx], [0, Ly], [Lz, Lz]),
        ([Lx, 0], [Ly, Ly], [Lz, Lz]),
        ([0, 0], [Ly, 0], [Lz, Lz]),
        # Vertical edges
        ([0, 0], [0, 0], [0, Lz]),
        ([Lx, Lx], [0, 0], [0, Lz]),
        ([Lx, Lx], [Ly, Ly], [0, Lz]),
        ([0, 0], [Ly, Ly], [0, Lz]),
    ]

    for edge_x, edge_y, edge_z in box_edges:
        fig.add_trace(go.Scatter3d(
            x=edge_x, y=edge_y, z=edge_z,
            mode='lines',
            line=dict(color='black', width=4),
            showlegend=False,
            hoverinfo='skip'
        ))

    # Configure layout with fixed axis ranges
    title_text = f"3D View - Iteration {conf_iter}<br><sub>Urp={energy_urp:.1f}, E_sp={energy_sp:.1f}, E_int={energy_int:.1f} kB·K</sub>"

    fig.update_layout(
        title=dict(text=title_text, x=0.5, font=dict(size=14)),
        scene=dict(
            xaxis_title="X (Å)", yaxis_title="Y (Å)", zaxis_title="Z (Å)",
            aspectmode='cube',
            bgcolor='rgba(240,240,240,0.1)',
            xaxis=dict(showgrid=True, gridcolor='lightgray', range=[0, Lx]),
            yaxis=dict(showgrid=True, gridcolor='lightgray', range=[0, Ly]),
            zaxis=dict(showgrid=True, gridcolor='lightgray', range=[0, Lz])
        ),
        showlegend=True, legend=dict(x=0.02, y=0.98),
        width=600, height=600
    )

    return fig


def plot_worldline_2d_xy(config_idx):
    """
    Plot the 2D XY projection with 3x3 PBC replication.
    Returns the figure object.
    """
    conf_iter = conf_indices[config_idx]
    conf = confs[conf_iter]
    
    # Get energy at this iteration
    log_row = log_data[log_data['Iteration'] == conf_iter]
    if len(log_row) > 0:
        energy_urp = log_row['Energy(Urp)'].values[0]
        energy_sp = log_row['E_sp'].values[0]
        energy_int = log_row['E_int'].values[0]
    else:
        energy_urp = energy_sp = energy_int = np.nan

    fig = go.Figure()

    Lx, Ly = box[0], box[1]
    
    # Wrap coordinates to [0, L]
    wrapped_coords = conf.beadCoord.copy()
    wrapped_coords[:, :, 0] = wrapped_coords[:, :, 0] % Lx
    wrapped_coords[:, :, 1] = wrapped_coords[:, :, 1] % Ly

    # Get normalized cycle lengths
    cycle_lengths_normalized = conf.cycleSizeDist / conf.numTimeSlices
    unique_cycle_lengths = np.unique(cycle_lengths_normalized)

    # Create color mapping
    cycle_length_to_color_idx = {length: idx for idx, length in enumerate(unique_cycle_lengths)}
    num_unique_lengths = len(unique_cycle_lengths)

    # Group beads by cycle length with 3x3 replication in XY
    beads_by_cycle_length = {length: {'x': [], 'y': []} for length in unique_cycle_lengths}

    # Replicate in 3x3 grid
    for offset_x in [-Lx, 0, Lx]:
        for offset_y in [-Ly, 0, Ly]:
            for m in range(conf.numTimeSlices):
                for n in range(conf.numParticles):
                    cycle_idx = conf.cycleIndex[m, n]
                    cycle_length = cycle_lengths_normalized[cycle_idx]
                    beads_by_cycle_length[cycle_length]['x'].append(wrapped_coords[m, n, 0] + offset_x)
                    beads_by_cycle_length[cycle_length]['y'].append(wrapped_coords[m, n, 1] + offset_y)

    # Plot each cycle length group
    for cycle_length in unique_cycle_lengths:
        color_idx = cycle_length_to_color_idx[cycle_length]
        color = f'hsl({(color_idx * 360 / num_unique_lengths) % 360}, 80, 50)'

        coords = beads_by_cycle_length[cycle_length]
        fig.add_trace(go.Scatter(
            x=coords['x'],
            y=coords['y'],
            mode='markers',
            marker=dict(size=2, color=color),
            name=f'cycle {cycle_length:.0f}'
        ))

    # Draw primary box boundaries (thick black lines)
    fig.add_trace(go.Scatter(
        x=[0, Lx, Lx, 0, 0],
        y=[0, 0, Ly, Ly, 0],
        mode='lines',
        line=dict(color='black', width=4),
        showlegend=False,
        hoverinfo='skip'
    ))

    # Draw replica box boundaries (thin gray lines)
    for offset_x in [-Lx, 0, Lx]:
        for offset_y in [-Ly, 0, Ly]:
            if offset_x == 0 and offset_y == 0:
                continue
            fig.add_trace(go.Scatter(
                x=[offset_x, offset_x + Lx, offset_x + Lx, offset_x, offset_x],
                y=[offset_y, offset_y, offset_y + Ly, offset_y + Ly, offset_y],
                mode='lines',
                line=dict(color='lightgray', width=1),
                showlegend=False,
                hoverinfo='skip',
                opacity=0.3
            ))

    # Configure layout
    title_text = f"2D XY Projection - Iteration {conf_iter}<br><sub>Urp={energy_urp:.1f}, E_sp={energy_sp:.1f}, E_int={energy_int:.1f} kB·K | 3×3 PBC replication</sub>"

    fig.update_layout(
        title=dict(text=title_text, x=0.5, font=dict(size=14)),
        xaxis=dict(title="X (Å)", showgrid=True, gridcolor='lightgray', range=[-Lx, 2*Lx], scaleanchor='y', scaleratio=1),
        yaxis=dict(title="Y (Å)", showgrid=True, gridcolor='lightgray', range=[-Ly, 2*Ly]),
        showlegend=True,
        width=600,
        height=600
    )

    return fig


print("✅ Visualization functions ready")

✅ Visualization functions ready


## Interactive Slider

In [15]:
def show_both_views(config_idx):
    """Display both 3D and 2D views side by side"""
    conf_iter = conf_indices[config_idx]
    
    # Print info
    print(f"🔍 Config at Iteration {conf_iter}")
    conf = confs[conf_iter]
    print(f"   M={conf.numTimeSlices}, N={conf.numParticles}")
    print(f"   Cycles: {conf.cycleSizeDist.shape[0]}")
    
    # Create figure objects
    fig_3d = plot_worldline_3d(config_idx)
    fig_2d = plot_worldline_2d_xy(config_idx)
    
    # Display side by side using HBox
    output_3d = Output()
    output_2d = Output()
    
    with output_3d:
        fig_3d.show()
    
    with output_2d:
        fig_2d.show()
    
    display(HBox([output_3d, output_2d]))

print("🎯 Interactive Minimization Trajectory Visualization")
print(f"Use slider to view {len(conf_indices)} saved configurations")

w = interact(
    show_both_views, 
    config_idx=IntSlider(
        min=0, 
        max=len(conf_indices)-1, 
        step=1, 
        value=0,
        description='Snapshot:'
    )
)
display(w)

🎯 Interactive Minimization Trajectory Visualization
Use slider to view 26 saved configurations


interactive(children=(IntSlider(value=0, description='Snapshot:', max=25), Output()), _dom_classes=('widget-in…

<function __main__.show_both_views(config_idx)>